# Docker로 PostgreSQL 띄우기 (시연 · Windows판)

> **`docker run` 한 줄로 내 PC에 아무것도 설치하지 않고 진짜 PostgreSQL 17 서버를 띄우고, `psql`로 접속해 데이터베이스를 만들고, Python(`psycopg`)에서 `.env`의 비밀번호로 안전하게 연결하는 것**까지 직접 실행하며 확인합니다.

## 차시 학습 목표
Docker 소개에서 **개념**으로 배운 컨테이너·이미지·볼륨·포트를 이제 **실제 명령**으로 옮깁니다.
- `docker run`으로 `db-pg`(PostgreSQL 17) 컨테이너를 기동합니다.
- `psql`로 접속해 메타명령(`\l`·`\dt`)을 쓰고 `CREATE DATABASE`로 DB를 만듭니다.
- 포트(`-p`)·볼륨(`-v`)·환경변수(`-e`)의 의미를 실측으로 이해합니다.
- Python에서 `psycopg`로, 비밀번호는 같은 폴더 `.env` 파일에서 `load_dotenv()`로 읽어 연결합니다.

## 다루는 내용
- 이미지 확인 & 컨테이너 기동
- psql 접속 & 메타명령 & DB 생성
- 포트·볼륨·환경변수 이해
- Python에서 연결 — psycopg & .env

## 실행 안내
- 🐳 **선행 조건**: Docker Desktop이 설치·기동되어 있어야 합니다. 이미지 `pgvector/pgvector:pg17`는 최초 1회 `pull`이 필요합니다(강사 PC는 준비 완료).
- 🪟 **이 노트북은 Windows용**입니다 — 셸 명령이 cmd 문법(`>nul`·`findstr`)으로 되어 있어요. macOS/Linux에서는 원본 `PostgreSQL_1_Docker.ipynb`를 쓰세요.
- 위에서 아래로 **순서대로** 실행하면 에러 없이 끝까지 돕니다. 저장된 출력은 원본(macOS) 실측 예시이며, Windows용으로 명령을 바꾼 셀은 출력 없이 제공됩니다(수업에서 직접 실행).
- 🔁 **재실행 안전**: 컨테이너·DB 생성 셀은 이미 있으면 건너뛰도록(멱등) 작성했습니다.
- 🔐 비밀번호는 코드에 **하드코딩하지 않고** 같은 폴더 `.env` 파일(`PGPASSWORD=...`)에서 `load_dotenv()`로 읽습니다.
- 📌 Docker의 pull·기동 시간, 컨테이너 ID 등 **타이밍류 값은 기기·네트워크마다 다릅니다**(출력의 시간·ID는 예시).

In [1]:
import os
os.system('chcp 65001')

0

## 이미지 확인 & 컨테이너 기동

- `docker images`로 PostgreSQL 이미지(`pgvector/pgvector:pg17`)가 준비됐는지 확인합니다.
- **`docker run` 한 줄**로 `db-pg` 컨테이너를 백그라운드 기동합니다 — 옵션 하나하나가 어제 배운 개념(이름·환경변수·포트·볼륨)입니다.
- `docker ps`·`docker logs`로 "서버가 접속을 받을 준비가 됐다"를 확인합니다.

In [2]:
# ✅ 포인트: 이미지 = 컨테이너를 찍어내는 '틀'. 이 이미지가 로컬에 있어야 바로 기동됩니다(없으면 run이 자동 pull).
# 💡 pgvector/pgvector:pg17 = 'PostgreSQL 17 + pgvector 확장'이 담긴 이미지 (이 시간은 확장을 안 켜고 순수 PostgreSQL로 사용)
!docker images pgvector/pgvector:pg17

IMAGE   ID             DISK USAGE   CONTENT SIZE   EXTRA


In [3]:
# ✅ 포인트: docker run 한 줄로 PostgreSQL 서버가 뜹니다. 옵션 = 어제 배운 개념 그대로:
#   -d(백그라운드) · --name db-pg(이름) · -e POSTGRES_PASSWORD(환경변수) · -p 5432:5432(포트) · -v db-pg-data:...(볼륨)
# 💡 재실행 안전: db-pg가 이미 있으면 새로 만들지 않고, 없을 때만 docker run 합니다.
# 🪟 Windows cmd: 출력 버리기는 /dev/null 대신 nul, echo는 따옴표 없이 씁니다.
!docker inspect db-pg >nul 2>&1 && echo db-pg 컨테이너가 이미 있습니다 - 재사용합니다. || docker run -d --name db-pg -e POSTGRES_PASSWORD=postgres -p 5432:5432 -v db-pg-data:/var/lib/postgresql/data pgvector/pgvector:pg17
# 멈춰 있을 수도 있으니 start로 실행 상태를 보장합니다(이미 실행 중이면 아무 일도 안 함).
!docker start db-pg >nul && echo db-pg 실행 상태 보장 완료

b9045c5dac14aae0e35314fd01f49bb3179788c8641e23b1ce09d8e2ac71bd00


Unable to find image 'pgvector/pgvector:pg17' locally
pg17: Pulling from pgvector/pgvector
9524156efd7a: Pulling fs layer
f67c9bdd808a: Pulling fs layer
b70609756d71: Pulling fs layer
039e6f9f9752: Pulling fs layer
d7b627e06eac: Pulling fs layer
f495088ca134: Pulling fs layer
1e12f8d900eb: Pulling fs layer
740ea3feff0f: Pulling fs layer
5ac7c444edb0: Pulling fs layer
11ec1b6ae2a0: Pulling fs layer
3d4e001e01a2: Pulling fs layer
8cb2f3b54f01: Pulling fs layer
9ef6b0a1be9e: Pulling fs layer
62ca04624b47: Pulling fs layer
ef21471f3a2b: Pulling fs layer
44438d8012af: Pulling fs layer
b70609756d71: Download complete
9524156efd7a: Download complete
f67c9bdd808a: Download complete
f495088ca134: Download complete
1e12f8d900eb: Download complete
d7b627e06eac: Download complete
740ea3feff0f: Download complete
5ac7c444edb0: Download complete
44438d8012af: Download complete
3d4e001e01a2: Download complete
8cb2f3b54f01: Download complete
11ec1b6ae2a0: Download complete
9ef6b0a1be9e: Download comple

db-pg 실행 상태 보장 완료


In [4]:
# ✅ 포인트: docker ps = 실행 중인 컨테이너 목록. STATUS가 Up, PORTS에 5432가 보이면 정상.
!docker ps --filter name=db-pg

CONTAINER ID   IMAGE                    COMMAND                  CREATED              STATUS              PORTS                                         NAMES
b9045c5dac14   pgvector/pgvector:pg17   "docker-entrypoint.s…"   About a minute ago   Up About a minute   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp   db-pg


In [5]:
# ✅ 포인트: docker logs에 "database system is ready to accept connections"가 보이면 서버 준비 완료.
# 💡 컨테이너가 접속을 받을 준비가 될 때까지 잠깐 기다립니다(최초 기동 시 몇 초 — 시간은 기기마다 다름).
# 🪟 Windows cmd: grep 대신 findstr — 매칭 줄이 여러 개면 맨 아래가 최신입니다.
import subprocess, time
for i in range(30):
    r = subprocess.run(["docker", "exec", "db-pg", "pg_isready", "-U", "postgres"],
                       capture_output=True, text=True)
    if "accepting connections" in r.stdout:
        print("준비 완료:", r.stdout.strip())
        break
    time.sleep(1)
!docker logs db-pg 2>&1 | findstr /C:"ready to accept connections"

준비 완료: /var/run/postgresql:5432 - accepting connections
2026-09-22 01:38:45.412 UTC [49] LOG:  database system is ready to accept connections
2026-09-22 01:38:45.694 UTC [1] LOG:  database system is ready to accept connections


### 정리
- **핵심**: `docker run -d --name db-pg -e POSTGRES_PASSWORD=postgres -p 5432:5432 -v db-pg-data:/var/lib/postgresql/data pgvector/pgvector:pg17` **한 줄**로 진짜 PostgreSQL 서버가 떴습니다. **설치가 아니라 실행**입니다.
- **흔한 실수**: 포트 `5432`가 이미 다른 PostgreSQL에 쓰이면 기동이 실패합니다 → 기존 서버를 끄거나 `-p 5433:5432`로 바꿉니다.

## psql 접속 & 메타명령 & DB 생성

- **`docker exec`** 로 컨테이너 안의 `psql`(DB 콘솔)에 접속합니다.
- **메타명령**(`\l` DB 목록 · `\dt` 테이블 목록 · `\d` 상세)과 **SQL**(`SELECT version();`)을 구분합니다.
- **`CREATE DATABASE`** 로 오늘 쓸 두 DB(`library` 시연용 · `hanbit_bank` 개인실습용)를 만듭니다.

In [6]:
# ✅ 포인트: docker exec -it db-pg psql -U postgres = 컨테이너 안 psql 콘솔에 접속.
#   (노트북에서는 -it 없이 -c "SQL"로 한 줄씩 실행합니다. 강의장 터미널에서는 -it로 대화형 접속)
# 💡 psql에서 \로 시작하면 psql 메타명령, 그냥 치면 SQL 입니다.
!docker exec db-pg psql -U postgres -c "SELECT version();"

                                                           version                                                            
------------------------------------------------------------------------------------------------------------------------------
 PostgreSQL 17.11 (Debian 17.11-1.pgdg12+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit
(1 row)



In [7]:
# ✅ 포인트: \l = 데이터베이스 목록. 기본 DB(postgres·template0·template1)에 오늘 만들 DB가 더해집니다.
!docker exec db-pg psql -U postgres -c "\l"

                                                    List of databases
   Name    |  Owner   | Encoding | Locale Provider |  Collate   |   Ctype    | Locale | ICU Rules |   Access privileges   
-----------+----------+----------+-----------------+------------+------------+--------+-----------+-----------------------
 postgres  | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | 
 template0 | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | =c/postgres          +
           |          |          |                 |            |            |        |           | postgres=CTc/postgres
 template1 | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | =c/postgres          +
           |          |          |                 |            |            |        |           | postgres=CTc/postgres
(3 rows)



In [8]:
# ✅ 포인트: CREATE DATABASE 로 새 데이터베이스를 만듭니다. 시연은 library, 개인실습은 hanbit_bank.
# 💡 재실행 안전: PostgreSQL엔 'CREATE DATABASE IF NOT EXISTS'가 없어, 있으면 건너뛰도록 조건 확인합니다.
# 🪟 Windows cmd: grep -q 대신 findstr ... >nul (찾으면 성공 → || 뒤 CREATE는 건너뜀)
!docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='library'" | findstr 1 >nul || docker exec db-pg psql -U postgres -c "CREATE DATABASE library"
!docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='hanbit_bank'" | findstr 1 >nul || docker exec db-pg psql -U postgres -c "CREATE DATABASE hanbit_bank"
!docker exec db-pg psql -U postgres -c "\l"

CREATE DATABASE
CREATE DATABASE
                                                     List of databases
    Name     |  Owner   | Encoding | Locale Provider |  Collate   |   Ctype    | Locale | ICU Rules |   Access privileges   
-------------+----------+----------+-----------------+------------+------------+--------+-----------+-----------------------
 hanbit_bank | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | 
 library     | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | 
 postgres    | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | 
 template0   | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | =c/postgres          +
             |          |          |                 |            |            |        |           | postgres=CTc/postgres
 template1   | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |

In [9]:
# ✅ 포인트: -d library = library DB에 접속, \dt = 테이블 목록. 아직 테이블이 없어 비어 있습니다(다음 시연에서 만듭니다).
!docker exec db-pg psql -U postgres -d library -c "\dt"

Did not find any relations.


### 정리
- **핵심**: `\l`(DB 목록)·`\dt`(테이블 목록)·`\d 테이블`(상세)은 **psql 메타명령**(`\`로 시작), `SELECT`·`CREATE DATABASE`는 **SQL**입니다.
- **흔한 실수**: `psql` 대화형에서 SQL 끝에 **세미콜론(`;`)을 빠뜨리면** 실행되지 않고 다음 줄을 기다립니다 — SQL은 `;`로 끝냅니다.

## 포트·볼륨·환경변수 이해

- **포트 매핑**(`-p 5432:5432`): 내 PC의 5432 → 컨테이너의 5432. 그래서 `localhost:5432`로 접속됩니다.
- **볼륨**(`-v db-pg-data:...`): 데이터는 컨테이너가 아니라 **볼륨**에 삽니다 → 컨테이너를 껐다 켜도, 심지어 지워도 데이터가 남습니다.
- **환경변수**(`-e POSTGRES_PASSWORD`): 이미지가 첫 기동 때 비밀번호를 설정합니다.

In [10]:
# ✅ 포인트: 포트 매핑 = 컨테이너 안(5432)과 내 PC(5432)를 잇는 '출입문'. 그래서 localhost:5432로 접속됩니다.
!docker port db-pg

5432/tcp -> 0.0.0.0:5432
5432/tcp -> [::]:5432


In [11]:
# ✅ 포인트: 데이터는 볼륨 db-pg-data 에 저장됩니다(컨테이너와 분리).
!docker volume ls --filter name=db-pg-data

DRIVER    VOLUME NAME
local     db-pg-data


In [12]:
# ✅ 포인트: 껐다(stop) 켜도(start) 데이터가 볼륨에 남아 있음을 확인합니다("데이터는 볼륨에 산다").
# 🪟 Windows cmd: 여러 단어 OR 검색은 findstr "단어1 단어2" (공백 = OR)
import subprocess, time
!docker stop db-pg && docker start db-pg
for i in range(30):                     # 재기동 후 준비 대기
    r = subprocess.run(["docker", "exec", "db-pg", "pg_isready", "-U", "postgres"],
                       capture_output=True, text=True)
    if "accepting connections" in r.stdout:
        break
    time.sleep(1)
# 재기동 후에도 library·hanbit_bank DB가 그대로 있으면 볼륨 영속화 성공
print("[재기동 후에도 남아 있는 DB]")
!docker exec db-pg psql -U postgres -c "\l" | findstr "library hanbit_bank"

db-pg
db-pg
[재기동 후에도 남아 있는 DB]
 hanbit_bank | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | 
 library     | postgres | UTF8     | libc            | en_US.utf8 | en_US.utf8 |        |           | 


### 정리
- **핵심**: **포트**=접속 통로, **볼륨**=데이터 창고, **환경변수**=초기 설정. 데이터는 컨테이너가 아니라 **볼륨**(`db-pg-data`)에 살아, 컨테이너를 지워도(`docker rm -f db-pg`) 볼륨만 있으면 데이터가 보존됩니다.
- **흔한 실수 / 주의**: 볼륨 자체를 지우면(`docker volume rm db-pg-data`) 데이터가 **영구 삭제**됩니다 — 실습에서는 지우지 않습니다.

## Python에서 연결 — psycopg & .env

- **`psycopg`**(v3) 드라이버로 Python에서 PostgreSQL에 연결합니다.
- 비밀번호는 코드에 **하드코딩하지 않고** 같은 폴더 `.env` 파일에서 `load_dotenv()`로 읽습니다.
- 연결 → 커서 → `SELECT version()` → 종료(`close`)의 기본 흐름을 확인합니다. 다음 시연은 이 연결로 도서관 DB를 이식합니다.

In [2]:
# ✅ 포인트: psycopg = Python용 PostgreSQL 드라이버. [binary]는 미리 빌드된 배포판이라 설치가 편합니다.
# 🆕 최신 문법: 오늘은 psycopg 'v3'를 씁니다(import psycopg). 과거의 psycopg2와 다릅니다.
%pip install -q "psycopg[binary]==3.3.4" python-dotenv
import psycopg
print("psycopg 버전:", psycopg.__version__)

Note: you may need to restart the kernel to use updated packages.
psycopg 버전: 3.3.4


In [3]:
# ✅ 포인트: 비밀번호를 코드에 쓰지 않습니다 — 같은 폴더의 .env 파일에 두고 load_dotenv()로 읽습니다.
# 💡 .env 내용은 PGPASSWORD=postgres 처럼 이름=값 한 줄씩 — Git에 올리지 않는 파일입니다!
import os
from dotenv import load_dotenv
load_dotenv()                                   # 같은 폴더의 .env → 환경변수
pw = os.environ["PGPASSWORD"]                   # 비밀번호는 .env 파일에 (하드코딩 금지)

# ✅ 포인트: psycopg.connect(host, port, dbname, user, password)로 연결합니다.
conn = psycopg.connect(host="localhost", port=5432, dbname="library", user="postgres", password=pw)
cur = conn.cursor()
print("연결 완료 →", conn)

연결 완료 → <psycopg.Connection [IDLE] (host=localhost user=postgres database=library) at 0x209a9590050>


In [4]:
# ✅ 포인트: cur.execute(SQL) → cur.fetchone()으로 결과 한 행을 가져옵니다.
cur.execute("SELECT version();")
print(cur.fetchone()[0])

PostgreSQL 17.11 (Debian 17.11-1.pgdg12+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


In [5]:
# ✅ 포인트: 다 쓰면 커서·연결을 닫습니다. (조회만 했으므로 commit은 불필요)
cur.close()
conn.close()
print("연결을 닫았습니다. 다음 시연에서 이 연결로 도서관 DB를 이식합니다.")

연결을 닫았습니다. 다음 시연에서 이 연결로 도서관 DB를 이식합니다.


## 마무리 — 오늘 세운 것과 다음 단계

`docker run` **한 줄**로 설치 없이 진짜 PostgreSQL 17 서버(`db-pg`)를 띄우고, `psql`로 접속해 `library`·`hanbit_bank` 데이터베이스를 만들고, Python(`psycopg`)에서 `.env`의 비밀번호로 안전하게 연결하는 것까지 확인했습니다.
- **기동**: `docker run`(포트·볼륨·환경변수) → `docker ps`·`docker logs`로 준비 확인
- **접속**: `docker exec ... psql` → 메타명령(`\l`·`\dt`) + `CREATE DATABASE`
- **영속화**: 데이터는 **볼륨**(`db-pg-data`)에 살아 컨테이너를 껐다 켜도 유지
- **연결**: `psycopg.connect(...)` + `.env`/`load_dotenv`(하드코딩 금지)

**➡️ 다음 시연 `PostgreSQL SQL 기초`**: 방금 만든 `library` DB에, **어제 sqlite3로 배운 도서관 SQL을 글자 하나 안 바꾸고** 이식해 "같은 SQL, 진짜 데이터베이스로"를 증명하고, SQLite와 갈리는 5개 지점을 짚습니다.

📌 컨테이너 정리는 `docker stop db-pg`(데이터는 볼륨에 유지)로 합니다.

## 종합 연습 — 쇼핑몰 DB를 PostgreSQL에 준비하기

오늘 배운 절차(psql · CREATE DATABASE · psycopg 접속 · 볼륨 영속)를 **쇼핑몰 DB(`shop`)** 로 복습합니다. `db-pg` 컨테이너가 실행 중이어야 해요(위 기동 셀 참고). 여기서 만든 `shop` DB는 **다음 시간 종합 연습에서 계속** 씁니다.

정답은 맨 아래 **정답 모음**에 한꺼번에 있어요.

In [ ]:
# 🧪 Q1 — psql로 shop 데이터베이스를 만드세요 (있으면 건너뛰는 재실행 안전 패턴으로) → \l 로 확인
# 💡 힌트: 위 'DB 생성' 셀의 pg_database 확인 패턴에서 이름만 shop 으로
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q2 — psycopg로 shop DB에 접속해, 지금 접속한 DB 이름을 SQL로 확인하세요
# 💡 힌트: psycopg.connect(..., dbname="shop", ...) → SELECT current_database()
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q3 — 컨테이너를 껐다 켜도 shop DB가 남아 있는지 확인하세요 (데이터는 볼륨에 산다!)
# 💡 힌트: docker stop/start → pg_isready 대기 → \l 에서 shop 찾기 (위 '볼륨 영속' 셀 패턴)
# 👇 아래 빈 셀에 직접 작성해 보세요

### 정답 모음 — 직접 푼 뒤에 확인하세요

In [ ]:
# ── Q1 정답 ──
# !docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='shop'" | findstr 1 >nul || docker exec db-pg psql -U postgres -c "CREATE DATABASE shop"
# !docker exec db-pg psql -U postgres -c "\l"
# # → 목록에 shop 이 보이면 성공 (재실행해도 오류 없이 건너뜁니다)

In [ ]:
# ── Q2 정답 ──
# conn_shop = psycopg.connect(host="localhost", port=5432, dbname="shop", user="postgres", password=pw)
# cur_shop = conn_shop.cursor()
# cur_shop.execute("SELECT current_database()")
# print(cur_shop.fetchone())
# # → ('shop',)

In [ ]:
# ── Q3 정답 ──
# import subprocess, time
# !docker stop db-pg && docker start db-pg
# for i in range(30):
#     r = subprocess.run(["docker", "exec", "db-pg", "pg_isready", "-U", "postgres"],
#                        capture_output=True, text=True)
#     if "accepting connections" in r.stdout:
#         break
#     time.sleep(1)
# !docker exec db-pg psql -U postgres -c "\l" | findstr shop
# # → shop 줄이 그대로 보이면 성공 — 컨테이너를 껐다 켜도 데이터는 볼륨에 남습니다
# # ⚠️ 재기동했으니 기존 연결(conn 등)은 끊겼습니다 — 다시 쓰려면 connect부터 다시!